# Registros

Registros (tipos de datos con campos con nombre). El libro usaba un módulo `record` de Python 2; aquí se usan `dataclasses`.

## 1. Personas

Definir `Persona` con nombre, DNI, edad y sexo (`'M'` mujer, `'V'` varón), una función que la muestre, otra que diga si es menor de edad y otra que devuelva su nombre de pila (la primera palabra del nombre).

In [1]:
from dataclasses import dataclass, field, replace

@dataclass
class Persona:
    nombre: str
    dni: str
    edad: int
    sexo: str = "M"

def mostrar_persona(p):
    print(f"Nombre: {p.nombre} | DNI: {p.dni} | Edad: {p.edad} | Sexo: {p.sexo}")

def es_menor(p):
    return p.edad < 18

def nombre_de_pila(p):
    return p.nombre.split()[0]

ana = Persona("Ana Pérez Ruiz", "12345678Z", 17, "M")
mostrar_persona(ana)
print(es_menor(ana), nombre_de_pila(ana))

Nombre: Ana Pérez Ruiz | DNI: 12345678Z | Edad: 17 | Sexo: M
True Ana


## 2. Lista de personas y la más vieja

Pedir datos de personas hasta que el usuario no quiera continuar y mostrar el nombre de la persona (o personas) de mayor edad.

In [2]:
personas = []
while True:
    personas.append(Persona(input("Nombre: "), input("DNI: "), int(input("Edad: ")), input("Sexo (M/V): ")))
    if input("¿Otra persona? (s/n): ") != "s":
        break
mayor = max(p.edad for p in personas)
print([p.nombre for p in personas if p.edad == mayor])

Nombre: Juan Paz
DNI: 1Z
Edad: 40
Sexo (M/V): V
¿Otra persona? (s/n): s
Nombre: Eva Gil
DNI: 2X
Edad: 52
Sexo (M/V): M
¿Otra persona? (s/n): s
Nombre: Luis Mar
DNI: 3Y
Edad: 52
Sexo (M/V): V
¿Otra persona? (s/n): n
['Eva Gil', 'Luis Mar']


## 3. Registros en memoria

¿Qué muestra este programa? `persona3 = persona2` es un alias (modifica a Pedro); `copia` crea un registro nuevo.

In [3]:
def copia(p):
    return replace(p)

def nada_util(persona1, persona2):
    persona1.edad += 1
    persona3 = persona2
    persona4 = copia(persona2)
    persona3.edad -= 1
    persona4.edad -= 2
    return persona4

juan = Persona("Juan Paz", "12345679Z", 19, "V")
pedro = Persona("Pedro López", "23456789D", 18, "V")
otro = nada_util(juan, pedro)
print(juan, pedro, otro, sep="\n")

Persona(nombre='Juan Paz', dni='12345679Z', edad=20, sexo='V')
Persona(nombre='Pedro López', dni='23456789D', edad=17, sexo='V')
Persona(nombre='Pedro López', dni='23456789D', edad=16, sexo='V')


## 4. Gestión de estudiantes

Cada estudiante tiene nombre, grupo (`A`, `B` o `C`), nota de examen (`None` si no se presentó) y si entregó la práctica. Aprueba quien se presenta, saca al menos 5 y entregó la práctica. Funciones para: listar ordenado alfabéticamente (manteniendo la lista siempre ordenada al insertar), listar un grupo, saber si un alumno aprobó, mostrar el acta con la nota de los presentados, porcentaje de aprobados sobre el total, alumnos sobre la media (con o sin práctica), media de un grupo y ordenar por nombre, por nota de examen y por calificación final (no presentados al final).

In [4]:
import bisect

@dataclass(order=True)
class Estudiante:
    nombre: str
    grupo: str = field(compare=False)
    examen: float | None = field(compare=False, default=None)
    practica: bool = field(compare=False, default=False)

def nota_final(e):
    if e.examen is None:
        return None
    return e.examen if e.practica else min(e.examen, 4)

def esta_aprobado(e):
    return e.examen is not None and e.examen >= 5 and e.practica

def alta(estudiantes, e):
    # insort mantiene el orden alfabético
    bisect.insort(estudiantes, e)

def listado_de_nombres(estudiantes, grupo=None):
    return [e.nombre for e in estudiantes if grupo is None or e.grupo == grupo]

def muestra_acta(estudiantes):
    for e in estudiantes:
        nota = "No presentado" if e.examen is None else f"{e.examen:4.1f} {'Aprobado' if esta_aprobado(e) else 'Suspenso'}"
        print(f"{e.nombre:8} {e.grupo} {nota}")

def porcentaje_aprobados(estudiantes):
    return 100 * sum(map(esta_aprobado, estudiantes)) / len(estudiantes)

def media_examen(estudiantes):
    notas = [e.examen for e in estudiantes if e.examen is not None]
    return sum(notas) / len(notas) if notas else 0

def sobre_la_media(estudiantes, con_practica=False):
    m = media_examen(estudiantes)
    return [e.nombre for e in estudiantes if e.examen is not None and e.examen > m and (e.practica or not con_practica)]

def media_grupo(estudiantes, grupo):
    return media_examen([e for e in estudiantes if e.grupo == grupo])

estudiantes = []
for datos in (("Pau", "A", 7.5, True), ("Ana", "B", 9.0, False), ("Iker", "A", None, True), ("Eva", "C", 5.5, True), ("Luis", "B", 3.0, True)):
    alta(estudiantes, Estudiante(*datos))
print(listado_de_nombres(estudiantes), listado_de_nombres(estudiantes, "A"))
muestra_acta(estudiantes)
print(porcentaje_aprobados(estudiantes), sobre_la_media(estudiantes), sobre_la_media(estudiantes, True), media_grupo(estudiantes, "B"))
print([e.nombre for e in sorted(estudiantes, key=lambda e: -1 if e.examen is None else e.examen)])
print([e.nombre for e in sorted(estudiantes, key=lambda e: (nota_final(e) is None, -(nota_final(e) or 0)))])

['Ana', 'Eva', 'Iker', 'Luis', 'Pau'] ['Iker', 'Pau']
Ana      B  9.0 Suspenso
Eva      C  5.5 Aprobado
Iker     A No presentado
Luis     B  3.0 Suspenso
Pau      A  7.5 Aprobado
40.0 ['Ana', 'Pau'] ['Pau'] 6.0
['Iker', 'Luis', 'Eva', 'Pau', 'Ana']
['Pau', 'Eva', 'Ana', 'Luis', 'Iker']


## 5. Colección de MP3 y álbumes

Tipo `MP3` (título, intérprete, duración, estilo) con `muestra_resumen_mp3` y `muestra_mp3`, y un menú para añadir canciones, listar estilos sin repetir, canciones de un intérprete o estilo, listar todo y eliminar por título e intérprete. Ampliarlo con álbumes (título, intérpretes y canciones) con alta, baja, listado ordenado por intérprete y búsqueda de canciones.

In [5]:
@dataclass
class MP3:
    titulo: str
    interprete: str
    duracion: int
    estilo: str

def muestra_resumen_mp3(c):
    print(f"{c.titulo} - {c.interprete}")

def muestra_mp3(c):
    print(f"Título: {c.titulo}\nIntérprete: {c.interprete}\nDuración: {c.duracion} s\nEstilo: {c.estilo}")

canciones = []
while (opcion := input("1 añadir 2 estilos 3 intérprete 4 estilo 5 todo 6 eliminar 7 salir: ")) != "7":
    if opcion == "1":
        canciones.append(MP3(input("Título: "), input("Intérprete: "), int(input("Duración: ")), input("Estilo: ")))
    elif opcion == "2":
        print(sorted({c.estilo for c in canciones}))
    elif opcion in ("3", "4"):
        valor = input("Valor: ")
        for c in canciones:
            if (c.interprete if opcion == "3" else c.estilo) == valor:
                muestra_resumen_mp3(c)
    elif opcion == "5":
        for c in canciones:
            muestra_mp3(c)
    elif opcion == "6":
        titulo, interprete = input("Título: "), input("Intérprete: ")
        canciones = [c for c in canciones if (c.titulo, c.interprete) != (titulo, interprete)]

@dataclass
class Album:
    titulo: str
    interpretes: list
    canciones: list = field(default_factory=list)

albumes = [Album("Kind of Blue", ["Miles Davis"], [MP3("So What", "Miles Davis", 562, "jazz")]),
           Album("Abbey Road", ["The Beatles"], [MP3("Something", "The Beatles", 182, "pop")])]
albumes.append(Album("Help!", ["The Beatles"], [MP3("Yesterday", "The Beatles", 125, "pop")]))
albumes = [a for a in albumes if a.titulo != "Abbey Road"]
for a in sorted(albumes, key=lambda a: a.interpretes):
    print(a.interpretes, a.titulo, len(a.canciones))
print([(a.titulo, c.titulo) for a in albumes for c in a.canciones if "yes" in c.titulo.lower()])

1 añadir 2 estilos 3 intérprete 4 estilo 5 todo 6 eliminar 7 salir: 1
Título: Yesterday
Intérprete: The Beatles
Duración: 125
Estilo: pop
1 añadir 2 estilos 3 intérprete 4 estilo 5 todo 6 eliminar 7 salir: 1
Título: So What
Intérprete: Miles Davis
Duración: 562
Estilo: jazz
1 añadir 2 estilos 3 intérprete 4 estilo 5 todo 6 eliminar 7 salir: 2
['jazz', 'pop']
1 añadir 2 estilos 3 intérprete 4 estilo 5 todo 6 eliminar 7 salir: 3
Valor: The Beatles
Yesterday - The Beatles
1 añadir 2 estilos 3 intérprete 4 estilo 5 todo 6 eliminar 7 salir: 4
Valor: jazz
So What - Miles Davis
1 añadir 2 estilos 3 intérprete 4 estilo 5 todo 6 eliminar 7 salir: 5
Título: Yesterday
Intérprete: The Beatles
Duración: 125 s
Estilo: pop
Título: So What
Intérprete: Miles Davis
Duración: 562 s
Estilo: jazz
1 añadir 2 estilos 3 intérprete 4 estilo 5 todo 6 eliminar 7 salir: 6
Título: Yesterday
Intérprete: The Beatles
1 añadir 2 estilos 3 intérprete 4 estilo 5 todo 6 eliminar 7 salir: 5
Título: So What
Intérprete: Mil

## 6. Biblioteca

Gestionar libros (título, autor, año) y socios (DNI, nombre, teléfono). Un socio puede tener 3 libros a la vez; si devuelve uno con más de 10 días, queda penalizado 3 días por cada día de retraso y no puede llevarse libros hasta entonces.

In [6]:
from datetime import date, timedelta

@dataclass
class Libro:
    titulo: str
    autor: str
    año: int
    prestado_a: str | None = None
    desde: date | None = None

@dataclass
class Socio:
    dni: str
    nombre: str
    telefono: str
    penalizado_hasta: date = date.min

class Biblioteca:
    def __init__(self):
        self.libros, self.socios = {}, {}

    def alta_libro(self, libro):
        self.libros[libro.titulo] = libro

    def baja_libro(self, titulo):
        if self.libros.get(titulo) and self.libros[titulo].prestado_a is None:
            del self.libros[titulo]

    def alta_socio(self, socio):
        self.socios[socio.dni] = socio

    def baja_socio(self, dni):
        if not any(l.prestado_a == dni for l in self.libros.values()):
            self.socios.pop(dni, None)

    def presta(self, titulo, dni, hoy):
        socio, libro = self.socios[dni], self.libros[titulo]
        if hoy < socio.penalizado_hasta:
            return f"{socio.nombre} está penalizado hasta el {socio.penalizado_hasta}"
        if sum(l.prestado_a == dni for l in self.libros.values()) >= 3:
            return "Ya tiene 3 libros"
        if libro.prestado_a:
            return "Libro no disponible"
        libro.prestado_a, libro.desde = dni, hoy
        return "Préstamo correcto"

    def devuelve(self, titulo, hoy):
        libro = self.libros[titulo]
        socio, retraso = self.socios[libro.prestado_a], (hoy - libro.desde).days - 10
        libro.prestado_a = libro.desde = None
        if retraso > 0:
            socio.penalizado_hasta = hoy + timedelta(days=3 * retraso)
            return f"Retraso de {retraso} días: penalizado hasta el {socio.penalizado_hasta}"
        return "Devolución correcta"

b = Biblioteca()
for t in ("Niebla", "Marianela", "Nada", "Tirano Banderas"):
    b.alta_libro(Libro(t, "Autor", 1900))
b.alta_socio(Socio("1Z", "Ana", "600"))
hoy = date(2026, 11, 1)
print([b.presta(t, "1Z", hoy) for t in ("Niebla", "Marianela", "Nada", "Tirano Banderas")])
print(b.devuelve("Niebla", hoy + timedelta(days=14)))
print(b.presta("Niebla", "1Z", hoy + timedelta(days=15)))

['Préstamo correcto', 'Préstamo correcto', 'Préstamo correcto', 'Ya tiene 3 libros']
Retraso de 4 días: penalizado hasta el 2026-11-27
Ana está penalizado hasta el 2026-11-27
